# Diffusion Models

Destory the data with noise over a thousand small steps.
Train one neural net to predict the noise, Reverse the process at inference. Today every mainstream image, video, 3D and music model runs on this loop, possibly with flow matching or consistentcy tricks on top.

## Problem Definition

# Build your Own

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(13)

T, T_DIM, HIDDEN = 40, 8, 24
STEPS, BATCH, LR = 4000, 64, 1e-2


class NoisePredictor(nn.Module):
    """eps_theta(x_t, t): predict the noise added at timestep t."""

    def __init__(self, x_dim, t_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, x_dim),
        )

    def forward(self, x_t, t_emb):
        return self.net(torch.cat([x_t, t_emb], dim=-1))


def sin_embed(t, steps, dim=8):
    """Sinusoidal timestep embedding. t: (B,) int/long."""
    half = dim // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=t.device).float() / max(half - 1, 1)))
    angles = t.float().unsqueeze(1) * freqs.unsqueeze(0)  # (B, half)
    return torch.cat([angles.sin(), angles.cos()], dim=-1)[:, :dim]


def make_schedule(steps):
    betas = torch.linspace(1e-4, 0.02, steps)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars


def sample_data(n):
    """1-D two-mode mixture: N(-2, 0.4^2) or N(+2, 0.4^2)."""
    mode = torch.randint(0, 2, (n,))
    mean = torch.where(mode == 0, -2.0, 2.0)
    return (mean + 0.4 * torch.randn(n)).unsqueeze(-1)


def q_sample(x0, t, alpha_bars, eps=None):
    """Forward process: x_t = sqrt(abar) x0 + sqrt(1-abar) eps."""
    if eps is None:
        eps = torch.randn_like(x0)
    abar = alpha_bars[t].unsqueeze(-1)
    return abar.sqrt() * x0 + (1.0 - abar).sqrt() * eps, eps


@torch.no_grad()
def p_sample(model, alphas, alpha_bars, betas, steps, t_dim, n=500):
    """Ancestral DDPM reverse chain."""
    x = torch.randn(n, 1)
    for t in range(steps - 1, -1, -1):
        t_batch = torch.full((n,), t, dtype=torch.long)
        eps_hat = model(x, sin_embed(t_batch, steps, t_dim))
        mean = (x - betas[t] / (1.0 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x.squeeze(-1)


def plot_real_vs_samples(reals, samples, bins=40):
    """Overlay real mixture vs DDPM samples; mark target modes at ±2."""
    reals = reals.detach().cpu().numpy().ravel()
    samples = samples.detach().cpu().numpy().ravel()
    lo, hi = -5.0, 5.0
    xs = np.linspace(lo, hi, 400)
    pdf = 0.5 * (
        np.exp(-0.5 * ((xs + 2) / 0.4) ** 2) / (0.4 * np.sqrt(2 * np.pi))
        + np.exp(-0.5 * ((xs - 2) / 0.4) ** 2) / (0.4 * np.sqrt(2 * np.pi))
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(reals, bins=bins, range=(lo, hi), density=True, alpha=0.45, label="real data")
    ax.hist(samples, bins=bins, range=(lo, hi), density=True, alpha=0.45, label="DDPM samples")
    ax.plot(xs, pdf, color="black", lw=1.5, label="true pdf")
    ax.axvline(-2, color="gray", ls="--", lw=1, alpha=0.7)
    ax.axvline(2, color="gray", ls="--", lw=1, alpha=0.7)
    y_top = ax.get_ylim()[1]
    ax.text(-2, y_top * 0.92, "mode A (−2)", ha="center", fontsize=9, color="gray")
    ax.text(2, y_top * 0.92, "mode B (+2)", ha="center", fontsize=9, color="gray")
    ax.set_xlim(lo, hi)
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    ax.set_title("1-D mixture: real data vs DDPM samples")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


betas, alphas, alpha_bars = make_schedule(T)
model = NoisePredictor(1, T_DIM, HIDDEN)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

print("=== training DDPM on two-mode 1-D mixture ===")
model.train()
for step in range(1, STEPS + 1):
    x0 = sample_data(BATCH)
    t = torch.randint(0, T, (BATCH,))
    x_t, eps = q_sample(x0, t, alpha_bars)
    eps_hat = model(x_t, sin_embed(t, T, T_DIM))
    loss = loss_fn(eps_hat, eps)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"step {step:5d}: loss {loss.item():.4f}")

print()
print("=== sampling ===")
model.eval()
samples = p_sample(model, alphas, alpha_bars, betas, T, T_DIM, n=500)
reals = sample_data(500).squeeze(-1)
plot_real_vs_samples(reals, samples)

pos = (samples > 0).sum().item()
print(f"mean {samples.mean():+.3f}, modeA(<0)={500 - pos}, modeB(>0)={pos}")

print()
print("takeaway: trained noise predictor + reverse chain reproduces both modes.")
print("          same loss function that scales to images, video, 3D.")
